# vLLM serving on Colab — single A100 80GB (LLM + Embedding + Reranker → 1 public API)

OpenAI-compatible **LLM + embedding + reranker** on **one** A100, fronted by a single-port
gateway and exposed through a **cloudflared** tunnel as one public `https://*.trycloudflare.com`
URL your backend can call.

This is the Colab counterpart of the repo's Docker stack (which targets 2× RTX 6000 Pro,
Blackwell). The A100 is **Ampere** — it cannot run the NVFP4 LLM — so the LLM is swapped to
the **same Qwen3-30B-A3B MoE quantized to int4 AWQ** (~18 GB). The embedding and reranker are
unchanged.

| Route (on the public URL) | Service | Model |
|---|---|---|
| `POST /v1/chat/completions` | LLM | Qwen3-30B-A3B-Instruct-2507-AWQ-4bit |
| `POST /v1/embeddings` | Embedding | BAAI/bge-m3 (1024-d) |
| `POST /v1/rerank` · `/v1/score` | Reranker | Qwen/Qwen3-Reranker-0.6B |

> **Requirements:** Runtime → *Change runtime type* → **A100 GPU** (Colab Pro/Pro+). ~22 GB of
> weights download on first run. **Colab sessions are ephemeral** — when the VM stops, the
> tunnel URL dies and models must re-download (mount Drive in cell 3 to cache them).

**Run the cells top-to-bottom.**

## 1 · Check the GPU
Confirm an A100 with ~80 GB is attached.

In [ ]:
import subprocess, sys
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
name = out.stdout.strip()
if 'A100' not in name:
    print('\n⚠️  Not an A100. This notebook is tuned for A100-SXM4-80GB.')
    print('    Runtime → Change runtime type → A100 GPU, then Runtime → Restart.')
if 'A100' in name and '40' in name.split(',')[-1]:
    print('\n⚠️  Looks like a 40 GB A100. Lower LLM_GPU_UTIL / LLM_MAX_MODEL_LEN in cell 3,')
    print('    or run fewer services — 3 models + long context will not fit in 40 GB.')

## 2 · Config
Edit `API_KEY` (and `HF_TOKEN` if you have one). Defaults match `colab/.env.colab.example`.
The three `*_GPU_UTIL` values sum to **0.71** → ~57/80 GB, leaving headroom so nothing OOMs.

In [ ]:
import os

# ── Auth ──────────────────────────────────────────────────────────────
os.environ['API_KEY']  = 'sk-secai2026'      # 🔑 CHANGE ME — backend must send this
os.environ['HF_TOKEN'] = ''                   # optional: faster/gated downloads

# ── Models ────────────────────────────────────────────────────────────
os.environ['LLM_MODEL']    = 'cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit'
os.environ['EMBED_MODEL']  = 'BAAI/bge-m3'
os.environ['RERANK_MODEL'] = 'Qwen/Qwen3-Reranker-0.6B'
os.environ['LLM_QUANTIZATION'] = 'awq_marlin'   # Ampere AWQ fast path
os.environ['LLM_TOOL_PARSER']  = 'hermes'
os.environ['LLM_MAX_MODEL_LEN'] = '32768'       # raise toward 262144 if needed (watch VRAM)

# ── VRAM split (must sum < 1.0) ───────────────────────────────────────
os.environ['LLM_GPU_UTIL']    = '0.60'
os.environ['EMBED_GPU_UTIL']  = '0.05'
os.environ['RERANK_GPU_UTIL'] = '0.06'

# ── Networking ────────────────────────────────────────────────────────
os.environ['LLM_PORT'] = '8001'; os.environ['EMBED_PORT'] = '8000'; os.environ['RERANK_PORT'] = '8002'
os.environ['GATEWAY_PORT'] = '8080'
os.environ['GATEWAY_REQUIRE_KEY'] = '1'    # public URL → require the Bearer key from backend
os.environ['HEALTH_TIMEOUT'] = '900'
os.environ['LOG_DIR'] = '/content/logs'

# ── (Optional) cache weights on Google Drive to skip re-downloading each session ──
USE_DRIVE_CACHE = False
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
    print('HF cache →', os.environ['HF_HOME'])
print('config set. API_KEY =', os.environ['API_KEY'])

## 3 · Install vLLM + gateway deps + cloudflared
Takes a few minutes (vLLM is large). **If you see a torch/numpy conflict warning, do**
**Runtime → Restart session once, then re-run cells 1–2 before continuing.**

In [ ]:
%pip install -q 'vllm>=0.10.2' 'huggingface_hub>=0.24' fastapi 'uvicorn[standard]' httpx

# cloudflared binary (quick tunnel, no signup)
import os, subprocess, urllib.request, stat
CF = '/usr/local/bin/cloudflared'
if not os.path.exists(CF):
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    urllib.request.urlretrieve(url, CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)
print(subprocess.run([CF, '--version'], capture_output=True, text=True).stdout.strip())

## 4 · Fetch the helper files
Clones this repo to get `colab/launch_vllm.py`, `colab/gateway.py`, and the reranker chat
template `templates/qwen3_reranker.jinja` (the reranker won't serve without it).

In [ ]:
import os
REPO_URL = 'https://github.com/DuyTa506/infra_serving_quick_start.git'
if not os.path.isdir('/content/infra_serving_quick_start'):
    !git clone --depth 1 $REPO_URL /content/infra_serving_quick_start
%cd /content/infra_serving_quick_start/colab
assert os.path.exists('../templates/qwen3_reranker.jinja'), 'reranker template missing'
print('helpers ready in', os.getcwd())

## 5 · Launch the 3 vLLM servers (sequential, health-gated)
Starts the LLM, waits until it's healthy, then the embedding, then the reranker — so each
reserves its VRAM slice before the next starts (this is what keeps a single card from
OOMing). **First run downloads ~22 GB**, so this can take 10–20 min. Watch progress with the
log-tail cell below.

In [ ]:
# Run the launcher in the background. It gates internally: starts the LLM, waits for its
# /health, THEN starts embedding, waits, THEN reranker — so VRAM is reserved in order and
# nothing OOMs. The next cell polls and shows each service going green.
import subprocess, os
os.makedirs(os.environ['LOG_DIR'], exist_ok=True)
launcher = subprocess.Popen(
    ['python', 'launch_vllm.py'],
    stdout=open(os.environ['LOG_DIR'] + '/launcher.log', 'ab'),
    stderr=subprocess.STDOUT,
)
print('launcher started (pid', launcher.pid, ') — now run the wait cell below')

In [ ]:
# Block until all three report /health 200 (sequential gate). Re-run if a server is slow.
import time, urllib.request
PORTS = {'llm': os.environ['LLM_PORT'], 'embedding': os.environ['EMBED_PORT'], 'reranker': os.environ['RERANK_PORT']}
TIMEOUT = int(os.environ['HEALTH_TIMEOUT']) * 3   # all three, generous for downloads
deadline = time.time() + TIMEOUT
done = set()
while time.time() < deadline and len(done) < 3:
    for name, port in PORTS.items():
        if name in done:
            continue
        try:
            with urllib.request.urlopen(f'http://localhost:{port}/health', timeout=5) as r:
                if r.status == 200:
                    print(f'✓ {name} healthy on :{port}'); done.add(name)
        except Exception:
            pass
    if len(done) < 3:
        time.sleep(5)
assert len(done) == 3, f'only {sorted(done)} came up — check the logs cell below'
print('\n✅ all three servers healthy')

In [ ]:
# Tail logs if something is slow/erroring. Switch the filename to embedding.log / reranker.log.
!tail -n 40 /content/logs/llm.log

## 6 · Start the single-port gateway (replaces nginx)
Fronts all three servers on `:8080`, routes by path, and injects the Bearer key toward vLLM.
Because the tunnel is public it also **requires** the backend's `Authorization: Bearer`.

In [ ]:
import subprocess, os, time, urllib.request
gw = subprocess.Popen(
    ['uvicorn', 'gateway:app', '--host', '0.0.0.0', '--port', os.environ['GATEWAY_PORT']],
    stdout=open(os.environ['LOG_DIR'] + '/gateway.log', 'ab'), stderr=subprocess.STDOUT,
)
time.sleep(4)
with urllib.request.urlopen(f"http://localhost:{os.environ['GATEWAY_PORT']}/health", timeout=10) as r:
    print('gateway /health →', r.status, r.read().decode())

## 7 · Open the cloudflared tunnel → public URL
Prints the `https://*.trycloudflare.com` URL. **Copy it into your backend.** The tunnel lives
as long as this cell's process (and the Colab session) does.

In [ ]:
import subprocess, re, time, os
port = os.environ['GATEWAY_PORT']
cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://localhost:{port}', '--no-autoupdate'],
                      stdout=open('/content/logs/cloudflared.log', 'ab'), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(40):
    time.sleep(2)
    try:
        log = open('/content/logs/cloudflared.log').read()
    except FileNotFoundError:
        continue
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m:
        PUBLIC_URL = m.group(0); break
assert PUBLIC_URL, 'tunnel URL not found — check /content/logs/cloudflared.log'
os.environ['PUBLIC_URL'] = PUBLIC_URL
print('\n' + '='*70)
print('  PUBLIC API URL  →  ' + PUBLIC_URL)
print('  API key (Bearer) →  ' + os.environ['API_KEY'])
print('='*70)

## 8 · Smoke test (chat + embedding + rerank via the public URL)

In [ ]:
import os, json, urllib.request
BASE = os.environ['PUBLIC_URL']; KEY = os.environ['API_KEY']
def call(path, payload):
    req = urllib.request.Request(BASE + path, data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {KEY}'})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)

chat = call('/v1/chat/completions', {'model': os.environ['LLM_MODEL'],
    'messages': [{'role': 'user', 'content': 'Say hi in one short sentence.'}], 'max_tokens': 64})
print('CHAT  :', chat['choices'][0]['message']['content'])

emb = call('/v1/embeddings', {'model': os.environ['EMBED_MODEL'], 'input': 'hello world'})
print('EMBED :', len(emb['data'][0]['embedding']), 'dims')

rr = call('/v1/rerank', {'model': os.environ['RERANK_MODEL'], 'query': 'capital of France',
    'documents': ['Paris is the capital of France.', 'Berlin is the capital of Germany.'], 'top_n': 2})
print('RERANK:', [(d['index'], round(d['relevance_score'], 3)) for d in rr['results']])

## 9 · Use it from your backend
Point your OpenAI-compatible client at the public URL and send the Bearer key:

```python
from openai import OpenAI
client = OpenAI(base_url='https://xxxx.trycloudflare.com/v1', api_key='sk-secai2026')

# chat
client.chat.completions.create(model='cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',
                               messages=[{'role':'user','content':'Hello!'}])
# embeddings
client.embeddings.create(model='BAAI/bge-m3', input='hello world')
```

```bash
# rerank (Cohere-style) — note: rerank/score are not in the openai SDK, call directly
curl https://xxxx.trycloudflare.com/v1/rerank \
  -H 'Authorization: Bearer sk-secai2026' -H 'Content-Type: application/json' \
  -d '{"model":"Qwen/Qwen3-Reranker-0.6B","query":"q","documents":["a","b"]}'
```

**Notes**
- The `trycloudflare.com` URL changes every time you re-run cell 7 / restart the session.
  For a stable URL, use a named cloudflared tunnel or ngrok with a reserved domain.
- Colab idle/usage limits will eventually stop the VM and kill the tunnel + models.
- GPU snapshot: run the cell below to confirm VRAM (~57 GB used, no OOM).

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader

### Teardown (stop everything)

In [ ]:
# Stop tunnel, gateway, and the 3 vLLM servers.
import json, os, signal
for p in ('cf', 'gw'):
    try: globals()[p].terminate()
    except Exception: pass
try:
    pids = json.load(open(os.environ['LOG_DIR'] + '/pids.json'))
    for name, pid in pids.items():
        try: os.kill(pid, signal.SIGTERM); print('stopped', name)
        except ProcessLookupError: pass
except FileNotFoundError:
    print('no pids.json — servers may already be stopped')